In [1]:
import pandas as pd

df = pd.read_csv("/Users/kaviraj/Desktop/GUVI/Project3/cleaned_data.csv")

df

,specialty,appointment_time,gender,no_show,disability,place,appointment_shift,age,under_12_years_old,over_60_years_old,...,storm_day_before,rain_intensity,heat_intensity,appointment_date_continuous,Hipertension,Diabetes,Alcoholism,Handcap,Scholarship,SMS_received
0,psychotherapy,17,F,yes,intellectual,Lake Marvinville,afternoon,9.0,1,0,...,1,no_rain,warm,2020-01-01,0,0,0,0,0,0
1,Unknown,7,M,no,intellectual,ITAPEMA,morning,11.0,1,0,...,1,no_rain,cold,2020-01-01,0,0,0,0,0,0
2,speech therapy,16,M,no,intellectual,ITAJAÍ,afternoon,8.0,1,0,...,1,no_rain,warm,2020-01-01,0,0,0,0,0,0
3,speech therapy,14,M,yes,intellectual,Sarahside,afternoon,9.0,1,0,...,1,moderate,mild,2020-01-01,0,0,0,0,0,1
4,physiotherapy,8,M,no,motor,ITAJAÍ,morning,12.0,0,0,...,1,no_rain,mild,2020-01-01,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109552,psychotherapy,16,M,yes,intellectual,Unknown,afternoon,18.0,0,0,...,1,no_rain,mild,2021-05-11,0,0,0,0,0,0
109553,speech therapy,9,F,yes,motor,Port Georgestad,morning,18.0,0,0,...,1,moderate,warm,2021-05-11,0,0,0,0,0,0
109554,psychotherapy,13,F,no,intellectual,Erinton,afternoon,8.0,1,0,...,1,moderate,mild,2021-05-11,0,0,0,0,0,1
109555,physiotherapy,8,F,no,motor,CAMBORIU,morning,7.0,1,0,...,1,no_rain,warm,2021-05-11,0,0,0,0,0,0


# Convert Date

In [2]:
import pandas as pd
import numpy as np

df['appointment_date_continuous'] = pd.to_datetime(
    df['appointment_date_continuous']
)

# Encode Rain & Heat Intensity

In [3]:
rain_map = {
    'no_rain':0,
    'weak':1,
    'moderate':2,
    'heavy':3
}

heat_map = {
    'warm':0,
    'mild':1,
    'cold':2,
    'heavy_warm':3,
    'heavy_cold':4
}

df['rain_intensity'] = df['rain_intensity'].map(rain_map)

df['heat_intensity'] = df['heat_intensity'].map(heat_map)

# Create Daily Demand Dataset

In [4]:
daily_df = df.groupby(
    'appointment_date_continuous'
).agg({

    'no_show':'count',

    'age':'mean',

    'under_12_years_old':'sum',

    'over_60_years_old':'sum',

    'SMS_received':'sum',

    'Hipertension':'sum',

    'Diabetes':'sum',

    'Alcoholism':'sum',

    'Scholarship':'sum',

    'average_temp_day':'mean',

    'average_rain_day':'mean',

    'max_temp_day':'mean',

    'max_rain_day':'mean',

    'rainy_day_before':'mean',

    'storm_day_before':'mean',

    'rain_intensity':'mean',

    'heat_intensity':'mean'

}).reset_index()


daily_df.rename(
    columns={
        'no_show':'daily_appointments'
    },
    inplace=True
)

# Gender Features

In [5]:
gender_daily = pd.crosstab(
    df['appointment_date_continuous'],
    df['gender']
)

gender_daily.columns = [
    f"gender_{col}"
    for col in gender_daily.columns
]

gender_daily = gender_daily.reset_index()

daily_df = daily_df.merge(
    gender_daily,
    on='appointment_date_continuous',
    how='left'
)

# Top Specialty Features

In [6]:
top_specialties = (
    df['specialty']
    .value_counts()
    .head(5)
    .index
)

for sp in top_specialties:

    temp = (
        df[df['specialty'] == sp]
        .groupby(
            'appointment_date_continuous'
        )
        .size()
        .reset_index(
            name=f"specialty_{sp}"
        )
    )

    daily_df = daily_df.merge(
        temp,
        on='appointment_date_continuous',
        how='left'
    )

In [7]:
# Fill empty spaces with 0 in speciality
daily_df.fillna(0, inplace=True)

# Feature Engineering

In [8]:
daily_df['day'] = daily_df['appointment_date_continuous'].dt.day

daily_df['day_of_week'] = (
    daily_df['appointment_date_continuous']
    .dt.day_name()
)

daily_df['month'] = (
    daily_df['appointment_date_continuous']
    .dt.month
)

onehot_cols = ['day_of_week']

daily_df = pd.get_dummies(
    daily_df,
    columns=onehot_cols,
    drop_first=True
)

# Lag Feature

In [9]:
# Number of appointments 1 day before
daily_df['lag_1'] = (
    daily_df['daily_appointments']
    .shift(1)
)

# Number of appointments 7 days before
daily_df['lag_7'] = (
    daily_df['daily_appointments']
    .shift(7)
)

# Number of appointments 30 days before
daily_df['lag_30'] = (
    daily_df['daily_appointments']
    .shift(30)
)

# Rolling Mean

In [10]:
# Average number of appointments during the previous 7 days (including current row)
daily_df['rolling_mean_7'] = (
    daily_df['daily_appointments']
    .shift(1)
    .rolling(7)
    .mean()
)

# Average appointments over the previous 30 days.
daily_df['rolling_mean_30'] = (
    daily_df['daily_appointments']
    .shift(1)
    .rolling(30)
    .mean()
)

In [11]:
# Filling lag and rolling_mean with 0 to maintain contiuous data
daily_df['lag_1'] = daily_df['lag_1'].fillna(0)
daily_df['lag_7'] = daily_df['lag_7'].fillna(0)
daily_df['lag_30'] = daily_df['lag_30'].fillna(0)

daily_df['rolling_mean_7'] = daily_df['rolling_mean_7'].fillna(0)
daily_df['rolling_mean_30'] = daily_df['rolling_mean_30'].fillna(0)

In [12]:
daily_df.isnull().sum().sort_values(ascending=False) 

appointment_date_continuous       0
day_of_week_Saturday              0
specialty_speech therapy          0
specialty_physiotherapy           0
specialty_Unknown                 0
specialty_occupational therapy    0
day                               0
month                             0
day_of_week_Monday                0
day_of_week_Sunday                0
gender_M                          0
day_of_week_Thursday              0
day_of_week_Tuesday               0
day_of_week_Wednesday             0
lag_1                             0
lag_7                             0
lag_30                            0
rolling_mean_7                    0
specialty_psychotherapy           0
gender_I                          0
daily_appointments                0
Scholarship                       0
age                               0
under_12_years_old                0
over_60_years_old                 0
SMS_received                      0
Hipertension                      0
Diabetes                    

In [15]:
print(daily_df.columns)
print(len(daily_df.columns))

Index(['appointment_date_continuous', 'daily_appointments', 'age',
       'under_12_years_old', 'over_60_years_old', 'SMS_received',
       'Hipertension', 'Diabetes', 'Alcoholism', 'Scholarship',
       'average_temp_day', 'average_rain_day', 'max_temp_day', 'max_rain_day',
       'rainy_day_before', 'storm_day_before', 'rain_intensity',
       'heat_intensity', 'gender_F', 'gender_I', 'gender_M',
       'specialty_psychotherapy', 'specialty_speech therapy',
       'specialty_physiotherapy', 'specialty_Unknown',
       'specialty_occupational therapy', 'day', 'month', 'day_of_week_Monday',
       'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday',
       'day_of_week_Tuesday', 'day_of_week_Wednesday', 'lag_1', 'lag_7',
       'lag_30', 'rolling_mean_7', 'rolling_mean_30'],
      dtype='object')
39


In [16]:
daily_df.to_csv("r_feature_engineering.csv", index=False)

print("Done")

Done
